# A gentle introduction to OOP

It often happens that we want to group data with the operations that can be performed on that data. This is where Object-Oriented Programming (OOP) sometimes happens to be useful.

OOP is a programming paradigm that uses objects to design applications and computer programs. It utilizes several techniques to design and implement the objects.

Say, we want to store information about an Employee. If we didn't know about custom object creation, we could choose to, say, use a tuple to store the information:
```python
employee = ('John Doe', 30, 50000, ROLE.DEVELOPER)
```
Clearly, this is not very readable.

Alternatively, we could use a dictionary

In [ ]:
from enum import IntEnum, auto

class Role(IntEnum):
    INTERN = auto()
    DEVELOPER = auto()
    TEAM_LEAD = auto()
    MANAGER = auto()


employee = {
    'name': 'John Doe',
    'age': 30,
    'salary': 50000,
    'role': Role.TEAM_LEAD
}

print(employee)

This is better, but accessing fields by strings is error-prone.

Besides, we might want to add some methods to the employee object, like `give_raise()` or `promote()`. We want an employee object to have a pretty string representation, and we want to be able to compare employees by their `role`. We could do all this with a dictionary, but it would be messy:

In [ ]:
def give_raise(employee: dict, amount: int) -> None:
    employee['salary'] += amount

def promote_one_level(employee: dict) -> None:
    employee['role'] = Role(employee['role'] + 1) if employee['role'] != Role.MANAGER else Role.MANAGER

def print_employee(employee: dict) -> None:
    print(f"Employee(name={employee['name']}, role={employee['role']}, age={employee['age']}, salary={employee['salary']})")

def is_greater_than(employee1: dict, employee2: dict) -> bool:
    return employee1['role'] > employee2['role']

def is_same_employee(employee1: dict, employee2: dict) -> bool:
    return (employee1['name'] == employee2['name'] and employee1['age'] == employee2['age'] and employee1['salary'] == employee['salary'] and employee1['role'] == employee2['role'])


This looks quite messy and unstructured. This is where OOP comes in. We can define a class `Employee` that will store the data and the methods that can be performed on that data:

In [ ]:
from enum import IntEnum, auto


class Role(IntEnum):
    INTERN = auto()
    DEVELOPER = auto()
    TEAM_LEAD = auto()
    MANAGER = auto()

    def __repr__(self) -> str:
        return f"{self.name.lower()}[{self.value}]"


class Employee:
    def __init__(self, name: str, age: int, salary: int, role: Role):
        self.name = name
        self.age = age
        self.salary = salary
        self.role = role

    def __repr__(self):
        return f"Employee(name={self.name}, role={self.role!r}, age={self.age}, salary={self.salary})"

    def __gt__(self, other: "Employee") -> bool:
        return self.role > other.role

    def __eq__(self, other: "Employee") -> bool:
        return (
            self.name == other.name
            and self.age == other.age
            and self.salary == other.salary
            and self.role == other.role
        )

    def give_raise(self, amount: int) -> None:
        self.salary += amount

    def promote_one_level(self) -> None:
        self.role = Role(min(self.role + 1, Role.MANAGER))

Now, this is much better. Not only does it logically group the data and operations together, but it also makes the code more readable and maintainable.

One can also define magic methods like `__str__`, `__eq__`, `__lt__`, etc., where it makes sense, to make the class more powerful, since then we can use built-in functions and operators (like `print()`, `==`, `<`, etc.) on the objects of the class.

In [ ]:
e1 = Employee("Alice", 30, 100_000, Role.INTERN)
e2 = Employee("Bob", 35, 80_000, Role.TEAM_LEAD)

In [ ]:
print(e1)

In [ ]:
e1 == e2

In [ ]:
e2 > e1

In [ ]:
e1.promote_one_level()
e1.give_raise(10_000)
print(e1)

Note that the undefined operators would raise an error:

In [ ]:
e1 / e2

Moreover, sometimes it makes sense to use data classes (e.g. [https://docs.python.org/3/library/dataclasses.html#module-dataclasses](dataclasses) module in Python) to define classes that are **mostly** containers for data. This way, we can avoid writing boilerplate code for the `__init__`, `__repr__`, `__eq__`, and other methods.

It's doing the rest of the work for us.

In [ ]:
from dataclasses import dataclass


@dataclass # literal magic
class EmployeeDC:
    name: str
    age: int
    salary: int
    role: Role

    def __gt__(self, other: "EmployeeDC") -> bool:
        return self.role > other.role

    def give_raise(self, amount: int) -> None:
        self.salary += amount

    def promote_one_level(self) -> None:
        self.role = Role(min(self.role + 1, Role.MANAGER))

In [ ]:
e1 = EmployeeDC("Alice", 30, 100_000, Role.INTERN)
e2 = EmployeeDC("Bob", 35, 80_000, Role.TEAM_LEAD)

In [ ]:
print(e1)

In [ ]:
e1 == e2

In [ ]:
e2 > e1

In [ ]:
e1.promote_one_level()
e1.give_raise(10_000)
print(e1)

Magic methods can define all possible operations on the objects of the class. 

The following is a custom class with many magic methods defined. You can think of it as some new number type having two real numbers as its components.

Take a look at these methods and find the invocations in the cells after the class definition.

In [ ]:
import math


class Something:
    def __init__(self, a: float, b: float):
        self.a = a
        self.b = b
    
    def __repr__(self): # repr(self)
        """Return a string representation of the object."""
        return f"Something(a={self.a}, b={self.b})"
    
    def __str__(self): # str(self) (called by `print`)
        """Return a human-readable string representation of the object."""
        return f"({self.a}, {self.b})" # this would look like a tuple
    
    def __add__(self, other: "Something") -> "Something": # self + other
        """Add two Something objects."""
        return Something(self.a + other.a, self.b + other.b)
    
    def __sub__(self, other: "Something") -> "Something": # self - other
        """Subtract two Something objects."""
        return Something(self.a - other.a, self.b - other.b)
    
    def __mul__(self, other: "Something") -> "Something": # self * other
        """Multiply two Something objects."""
        return Something(
            self.a*other.a - self.b*other.b,
            self.a*other.b + self.b*other.a
        )
    
    def __truediv__(self, other: "Something") -> "Something": # self / other
        """Divide two Something objects."""
        denominator = other.a**2 + other.b**2
        return Something(
            (self.a*other.a + self.b*other.b) / denominator,
            (self.b*other.a - self.a*other.b) / denominator
        )
    
    def __eq__(self, other: "Something") -> bool: # self == other
        """Check if two Something objects are equal."""
        return math.isclose(self.a, other.a) and math.isclose(self.b, other.b)
    
    def __ne__(self, other: "Something") -> bool: # self != other
        """Check if two Something objects are not equal."""
        return not self == other
    
    def __abs__(self) -> float: # abs(self)
        """Return the magnitude of the object."""
        return math.sqrt(self.a**2 + self.b**2)
    
    def __neg__(self) -> "Something": # -self
        """Return the negation of Something."""
        return Something(-self.a, -self.b)
    
    def __pos__(self) -> "Something": # +self
        """Return the positive version of Something."""
        return self
    
    def __invert__(self) -> "Something": # ~self
        """Return the conjugate of Something."""
        return Something(self.a, -self.b)
    
    def __pow__(self, power: int) -> "Something": # self ** power
        """Raise Something to an integer power."""
        result = Something(1, 0)
        for _ in range(power):
            result *= self
        return result
    
    def __bool__(self) -> bool: # bool(self) (called by `if self: ...`)
        """Return True if the object is not 'zero'."""
        return self.a != 0 or self.b != 0
    
    # some more magic methods (however, don't make much sense in this example)

    def __len__(self) -> int: # len(self)
        """Return the length of the object."""
        return 2
    
    def __getitem__(self, index: int) -> float: # self[index]
        """Return the value at the given index."""
        return (self.a, self.b)[index]
    
    def __setitem__(self, index: int, value: float) -> None: # self[index] = value
        """Set the value at the given index."""
        print(f"Setting index {index} to {value} of {self}")
    
    def __iter__(self): # iter(self) (called by `for x in self: ...`)
        """Return an iterator over the object."""
        return iter((self.a, self.b))
    
    def __reversed__(self): # reversed(self)
        """Return a reversed version of the object."""
        return Something(self.b, self.a)
    
    def __contains__(self, value: float): # value in self
        """Check if the object contains the given value."""
        return f"called {value} in {self}"
    
    def __call__(self, x: float): # self(x)
        """Call Something as a function."""
        return f"called {self} with {x}"
    
    def __matmul__(self, other): # self @ other
        """Another multiplication operation."""
        return f"called {self} @ {other}"
        

In [ ]:
c0 = Something(0, 0)
c1 = Something(1, 2)
c2 = Something(3, 4)

In [ ]:
c_sum = c1 + c2
print(c_sum)

c_diff = c1 - c2
print(c_diff)

c_prod = c1 * c2
print(c_prod)

c_quot = c1 / c2
print(c_quot)

In [ ]:
c_abs = abs(c1)
print(c_abs)

c_neg = -c1
print(c_neg)

c_pos = +c1
print(c_pos)

c_inv = ~c1
print(c_inv)

In [ ]:
c_pow = c1**3
print(c_pow)

c_bool = bool(c1)
print(c_bool)

In [ ]:
c_len = len(c1)
print(c_len)

c_getitem = c1[0]
print(c_getitem)

c_call = c1(5)
print(c_call)

c_matmul = c1 @ "something"
print(c_matmul)

Further reading: [A Guide to Python's Magic Methods](https://rszalski.github.io/magicmethods/) by Rafe Kettler.

## Problem 1. Basics

In the next cell you'll find a beginner's solution to the problem of storing information about a company. Rewrite it using a (data)class.

In [ ]:
def print_company(company: tuple[str, int]) -> None:
    """Print the name and number of employees of a company."""
    name, num_employees = company
    print(f"Company({name=!r}, {num_employees=})")


def is_bigger(company1: tuple[str, int], company2: tuple[str, int]) -> bool:
    """Return True if company1 has more employees than company2, False otherwise."""
    return company1[1] > company2[1]


def are_same(company1: tuple[str, int], company2: tuple[str, int]) -> bool:
    """Return True if company1 and company2 have the same name and number of employees, False otherwise."""
    return company1 == company2


def hire_one(company: tuple[str, int]) -> tuple[str, int]:
    """Return a new company with one more employee than the input company."""
    name, num_employees = company
    return (name, num_employees + 1)


company1 = ("Google", 100_000)
company2 = ("Facebook", 50_000)

print_company(company1)
print_company(company2)

print(is_bigger(company1, company2))

print(are_same(company1, company2))

company1_bigger = hire_one(company1)
print_company(company1_bigger)



In [ ]:
class Company:
    ...

In [ ]:
comp1 = Company("Google", 100_000)
comp2 = Company("Facebook", 50_000)
comp1_copy = Company("Google", 100_000)

In [ ]:
print(comp1) # calls __str__ (__repr__ if __str__ is not defined)
print(comp2)

In [ ]:
print(comp1 > comp2) # calls __gt__
print(comp1 == comp2) # calls __eq__
print(comp1 == comp1_copy)

In [ ]:
comp1_hired = comp1.hire_one()
print(comp1_hired)

## Problem 2. Defining operations

Write a class `Point2D` that stores the coordinates of a point (or a vector) on a euclidean plane.

The tests below should pass.

In [ ]:
from math import sqrt


# ?
class Point2D:
    ...


In [ ]:
# test set 1
p1 = Point2D(3, 4)
assert p1.x == 3 and p1.y == 4
p2 = Point2D(1.6, 4.2)
assert str(p2) == "Point2D(x=1.6, y=4.2)"

In [ ]:
from math import isclose


# test set 2
p3 = p1 + p2
assert isclose(p3.x, 4.6) and isclose(p3.y, 8.2)

p4 = p1 - p2
assert isclose(p4.x, 1.4) and isclose(p4.y, -0.2)

p5 = p1 * 2
assert p5 == Point2D(6, 8)

p6 = p2 / 10
assert isclose(p6.x, 0.16) and isclose(p6.y, 0.42)

In [ ]:
# test set 3
assert abs(p1) == 5
p1_norm = p1.normalized()
assert isclose(p1_norm.x, 0.6) and isclose(p1_norm.y, 0.8)

## Problem 3. Serialization

Implement methods for a class `Entry` such that it can be _serialized_ to and _deserialized_ from a string.

In [ ]:
import datetime
# import ???
from dataclasses import dataclass


@dataclass
class Entry:
    action: str
    code: int
    date_created: datetime.datetime
    ids: list[int]

    def serialize(self) -> str:
        ...

    @classmethod # <- look it up (and also @staticmethod)
    def deserialize(cls, entry_data: str) -> "Entry":
        ...

In [ ]:
# intended usage
e1 = Entry("open", 32, datetime.datetime(2013, 2, 14, 12, 32), [32, 5])
print(e1) # note that dataclass provides __repr__ and __str__

In [ ]:
entry_data = e1.serialize()
print(entry_data)

In [ ]:
e1_deserialized = Entry.deserialize(entry_data)
print(e1_deserialized)

In [ ]:
# serialization and deserialization should be inverses
assert e1 == e1_deserialized # note that dataclass provides __eq__

## Problem 4. Fractions

Write a class `MyFraction` to represent rational numbers.

In [ ]:
def gcd(a: int, b: int) -> int:
    """
    Return the greatest common divisor of a and b.

    >>> gcd(12, 15)
    3
    >>> gcd(12, 16)
    4
    >>> gcd(12, 17)
    1
    """
    ...


class MyFraction:
    def __init__(self, numerator: int, denominator: int) -> None:
        #* raise ValueError if the denominator is zero
        ...

    def __add__(self, other: "MyFraction") -> "MyFraction":
        ...

    def __sub__(self, other: "MyFraction") -> "MyFraction":
        ...

    def __mul__(self, other: "MyFraction") -> "MyFraction":
        ...

    def __truediv__(self, other: "MyFraction") -> "MyFraction":
        ...

    def __eq__(self, other: "MyFraction") -> bool:
        ...

    def __gt__(self, other: "MyFraction") -> bool:
        ...

    def __str__(self) -> str:
        ...

    def __repr__(self) -> str:
        ...

    def __float__(self) -> float:
        ...

    def as_tuple(self) -> tuple[int, int]:
        ...

    @classmethod
    def from_string(cls, fraction_str: str) -> "MyFraction":
        # assume the string is in the format "numerator/denominator" (as returned by __str__)
        ...

In [ ]:
# test gcd
assert gcd(12, 15) == 3
assert gcd(12, 16) == 4
assert gcd(12, 17) == 1
assert gcd(5, 7) == 1

In [ ]:
# test MyFraction: arithmetic operations and comparisons

assert MyFraction(6, 16) == MyFraction(3, 8)

assert MyFraction(1, 2) + MyFraction(1, 3) == MyFraction(5, 6)
assert MyFraction(1, 2) - MyFraction(1, 3) == MyFraction(1, 6)

assert MyFraction(1, 3) - MyFraction(1, 3) == MyFraction(0, 1)
assert MyFraction(-2, 4) + MyFraction(1, 2) == MyFraction(0, 1)

assert MyFraction(-4, 3) + MyFraction(1, 3) == MyFraction(-1, 1)

assert MyFraction(3, 4) * MyFraction(2, 3) == MyFraction(1, 2)
assert MyFraction(3, 4) / MyFraction(2, 3) == MyFraction(9, 8)

assert MyFraction(5, 6) > MyFraction(4, 5)

In [ ]:
import math

# test MyFraction: other
assert str(MyFraction(3, 4)) == "3/4"
assert repr(MyFraction(3, 4)) == "Fraction(3, 4)"
assert math.isclose(float(MyFraction(3, 4)), 0.75)

assert MyFraction(3, 4).as_tuple() == (3, 4)

In [ ]:
# test MyFraction: zero denominator
try:
    MyFraction(1, 0)
except ValueError:
    print('ok')
else:
    raise AssertionError("Expected a ValueError")

In [ ]:
# test MyFraction: serialization
f1 = MyFraction(3, 4)

f1_str = str(f1)
f1_deserialized = MyFraction.from_string(f1_str)

assert f1 == f1_deserialized

f2 = MyFraction(-8, 12)

f2_str = str(f2)
f2_deserialized = MyFraction.from_string(f2_str)

assert f2 == f2_deserialized